# fc2 embeddings + clasificadores ML — fold 2

Extracción de embeddings **fc2 (512-D)** desde la cabeza SPECT congelada aplicada a `f_hat = UNet_fold2(f_mri)`, entrenamiento de clasificadores clásicos (KNN, SVM, LR, DNN) en **3 clases** (control / PD NHY1 / PD NHY2) y visualización del espacio embebido en test.

**Script:** `scripts/train_fc2_ml_classifiers_fold2.py`  
**Resultados:** `result/.../ownKFold/models/fold2/embeddings/`

In [ ]:
from __future__ import annotations

import json
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

FST_ROOT = Path("/workspace/IL-FST")
EMB_DIR = FST_ROOT / "result" / "fst_clsfeat_mnet2d_v2_z78_112" / "ownKFold" / "models" / "fold2" / "embeddings"
SCRIPT = FST_ROOT / "scripts" / "train_fc2_ml_classifiers_fold2.py"

LABEL_NAMES = {0: "Control", 1: "PD NHY1", 2: "PD NHY2"}
STAGE_COLORS = {0: "#1f77b4", 1: "#ff7f0e", 2: "#d62728"}
STAGE_MARKERS = {0: "o", 1: "^", 2: "s"}

print("embeddings dir:", EMB_DIR)

## 1. Entrenar clasificadores (si aún no se ejecutó)

In [ ]:
if not (EMB_DIR / "best_model.joblib").is_file():
    subprocess.run([sys.executable, str(SCRIPT)], check=True)
else:
    print("Artifacts found; skipping training. Delete embeddings/ to re-run.")

## 2. Resultados del mejor modelo

In [ ]:
comparison = pd.read_csv(EMB_DIR / "model_comparison.csv")
metrics = json.loads((EMB_DIR / "metrics_fold2.json").read_text())
config = json.loads((EMB_DIR / "config.json").read_text())

print("Best model:", config["best_model"])
print(json.dumps(metrics["test"], indent=2))
print("\n", (EMB_DIR / "classification_report_fold2.txt").read_text())
comparison.sort_values(["selected", "val_macro_recall"], ascending=[False, False])

## 3. Visualización 2D / 3D del espacio fc2 (test)

Promedio de embeddings fc2 por sujeto en test; proyección con PCA (3D) y t-SNE (2D).

In [ ]:
embeddings = np.load(EMB_DIR / "fc2_embeddings.npy")
idx_df = pd.read_csv(EMB_DIR / "embeddings_index.csv", dtype={"subject_id": str})
test_slices = idx_df[idx_df["split"] == "test"].copy()

rows = []
for sid, g in test_slices.groupby("subject_id"):
    pos = g.index.values
    rows.append(
        {
            "subject_id": sid,
            "label_3class": int(g.label_3class.iloc[0]),
            "stratum": g.stratum.iloc[0],
            "emb": embeddings[pos].mean(axis=0),
        }
    )
subj_emb = pd.DataFrame(rows)
X = np.stack(subj_emb.emb.values)
y = subj_emb.label_3class.values.astype(int)
print(f"test subjects: {len(subj_emb)}  embedding dim: {X.shape[1]}")

In [ ]:
def scatter_by_class(ax, coords, labels, title, *, dim=2):
    for c in sorted(LABEL_NAMES):
        m = labels == c
        if not m.any():
            continue
        kw = dict(s=80, c=STAGE_COLORS[c], marker=STAGE_MARKERS[c], alpha=0.9, edgecolors="white", linewidths=0.4, label=LABEL_NAMES[c])
        if dim == 2:
            ax.scatter(coords[m, 0], coords[m, 1], **kw)
        else:
            ax.scatter(coords[m, 0], coords[m, 1], coords[m, 2], **kw)
    ax.set_title(title)
    ax.legend(loc="best", fontsize=9)


# 2D t-SNE
tsne = TSNE(n_components=2, perplexity=min(15, len(X) - 1), random_state=42, init="pca", learning_rate="auto")
X_tsne = tsne.fit_transform(X)

fig, ax = plt.subplots(figsize=(7, 6))
scatter_by_class(ax, X_tsne, y, "t-SNE fc2 (test, mean per subject)")
ax.set_xlabel("t-SNE 1")
ax.set_ylabel("t-SNE 2")
plt.tight_layout()
fig.savefig(EMB_DIR / "tsne_fc2_test_subjects.png", dpi=150)
plt.show()

In [ ]:
# 3D PCA
pca = PCA(n_components=3, random_state=42)
X_pca = pca.fit_transform(X)

fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection="3d")
scatter_by_class(ax, X_pca, y, "PCA fc2 (test, mean per subject)", dim=3)
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%})")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%})")
ax.set_zlabel(f"PC3 ({pca.explained_variance_ratio_[2]:.1%})")
plt.tight_layout()
fig.savefig(EMB_DIR / "pca3d_fc2_test_subjects.png", dpi=150)
plt.show()

In [ ]:
# Slice-level t-SNE on test (all slices, colored by class)
X_sl = embeddings[test_slices.index.values]
y_sl = test_slices.label_3class.values.astype(int)
tsne_sl = TSNE(n_components=2, perplexity=30, random_state=42, init="pca", learning_rate="auto")
X_tsne_sl = tsne_sl.fit_transform(X_sl)

fig, ax = plt.subplots(figsize=(7, 6))
scatter_by_class(ax, X_tsne_sl, y_sl, "t-SNE fc2 (test, all slices)")
ax.set_xlabel("t-SNE 1")
ax.set_ylabel("t-SNE 2")
plt.tight_layout()
fig.savefig(EMB_DIR / "tsne_fc2_test_slices.png", dpi=150)
plt.show()